In [1]:
from pyspark.sql import(
    functions as f, 
    SparkSession, 
    types as t
)

In [2]:
spark = SparkSession.builder.appName("df_most_popular").getOrCreate()

In [3]:
csv_file_path = "file:///home/jovyan/work/sample/hero-network.csv"
df = spark.read\
            .option('header','true')\
            .option('inferSchema','true')\
            .csv(csv_file_path)

In [4]:
df.show(10)

+--------------------+--------------------+
|               hero1|               hero2|
+--------------------+--------------------+
|       LITTLE, ABNER|      PRINCESS ZANDA|
|       LITTLE, ABNER|BLACK PANTHER/T'CHAL|
|BLACK PANTHER/T'CHAL|      PRINCESS ZANDA|
|       LITTLE, ABNER|      PRINCESS ZANDA|
|       LITTLE, ABNER|BLACK PANTHER/T'CHAL|
|BLACK PANTHER/T'CHAL|      PRINCESS ZANDA|
|STEELE, SIMON/WOLFGA|    FORTUNE, DOMINIC|
|STEELE, SIMON/WOLFGA| ERWIN, CLYTEMNESTRA|
|STEELE, SIMON/WOLFGA|IRON MAN/TONY STARK |
|STEELE, SIMON/WOLFGA|IRON MAN IV/JAMES R.|
+--------------------+--------------------+
only showing top 10 rows



In [5]:
data = df.groupBy('hero1')\
            .agg(
                f.collect_set('hero2').alias('connection')
            ).withColumnRenamed('hero1','hero')
data.show()

+--------------------+--------------------+
|                hero|          connection|
+--------------------+--------------------+
|             ABCISSA|[ELSIE DEE, FURY,...|
|ABSORBING MAN | MUTA|[DRAX | MUTANT X-...|
|ABSORBING MAN/CARL C|[SOMMERS, APRIL, ...|
|    ADAMSON, REBECCA|[KABALLA, GOLEM I...|
|   ADVENT/KYLE GROBE|[JUSTICE II/VANCE...|
|      AGAMEMNON III/|[ASTER, LUCIAN, H...|
|            AGAMOTTO|[MUNIPOOR, DORMAM...|
|             AGGAMON|[DR. STRANGE/STEP...|
|              AGINAR|[SIF, REJECT/RAN-...|
|                AGON|[MARISTA, BLACK B...|
|               AINET|[STORM/ORORO MUNR...|
|    AKUTAGAWA, OSAMU|[HUMAN TORCH/JOHN...|
|ALDEN, PROF. MEREDIT|[CABE, BETHANY, S...|
|             ALISTRO|[ENCHANTRESS/AMOR...|
|       ALVAREZ, PAUL|[ATOR, GENERAL, Z...|
|   AMERICAN SAMURAI/|[PAGE, KAREN, DAR...|
|             AMPERE/|[QUICKSILVER/PIET...|
|           ANCESTOR/|[RECORDER II, FOU...|
|ANCIENT ONE/BARON MO|[BLOODSTORM | MUT...|
|    ANDERSSEN, TANYA|[KA-ZAR/KE

In [6]:
data = data.withColumn('connection',f.concat_ws(',',f.col('connection')))
data.show()

+--------------------+--------------------+
|                hero|          connection|
+--------------------+--------------------+
|             ABCISSA|ELSIE DEE,FURY, C...|
|ABSORBING MAN | MUTA|DRAX | MUTANT X-V...|
|ABSORBING MAN/CARL C|SOMMERS, APRIL,HE...|
|    ADAMSON, REBECCA|KABALLA,GOLEM III...|
|   ADVENT/KYLE GROBE|JUSTICE II/VANCE ...|
|      AGAMEMNON III/|ASTER, LUCIAN,HOG...|
|            AGAMOTTO|MUNIPOOR,DORMAMMU...|
|             AGGAMON|DR. STRANGE/STEPHEN |
|              AGINAR|SIF,REJECT/RAN-SA...|
|                AGON|MARISTA,BLACK BOL...|
|               AINET|STORM/ORORO MUNRO...|
|    AKUTAGAWA, OSAMU|HUMAN TORCH/JOHNN...|
|ALDEN, PROF. MEREDIT|CABE, BETHANY,STO...|
|             ALISTRO|ENCHANTRESS/AMORA...|
|       ALVAREZ, PAUL|ATOR, GENERAL,ZAR...|
|   AMERICAN SAMURAI/|PAGE, KAREN,DARED...|
|             AMPERE/|QUICKSILVER/PIETR...|
|           ANCESTOR/|RECORDER II,FOUND...|
|ANCIENT ONE/BARON MO|BLOODSTORM | MUTA...|
|    ANDERSSEN, TANYA|KA-ZAR/KEV

In [8]:
data.coalesce(1).write.mode('overwrite').option('header', True).csv('file:///home/jovyan/work/output')


In [10]:
csv_file_path = "file:///home/jovyan/work/output"
df = spark.read\
            .option('header','true')\
            .option('inferSchema','true')\
            .csv(csv_file_path)
df.show()

+--------------------+--------------------+
|                hero|          connection|
+--------------------+--------------------+
|             ABCISSA|ELSIE DEE,FURY, C...|
|ABSORBING MAN | MUTA|DRAX | MUTANT X-V...|
|ABSORBING MAN/CARL C|SOMMERS, APRIL,HE...|
|    ADAMSON, REBECCA|KABALLA,GOLEM III...|
|   ADVENT/KYLE GROBE|JUSTICE II/VANCE ...|
|      AGAMEMNON III/|ASTER, LUCIAN,HOG...|
|            AGAMOTTO|MUNIPOOR,DORMAMMU...|
|             AGGAMON| DR. STRANGE/STEPHEN|
|              AGINAR|SIF,REJECT/RAN-SA...|
|                AGON|MARISTA,BLACK BOL...|
|               AINET|STORM/ORORO MUNRO...|
|    AKUTAGAWA, OSAMU|HUMAN TORCH/JOHNN...|
|ALDEN, PROF. MEREDIT|CABE, BETHANY,STO...|
|             ALISTRO|ENCHANTRESS/AMORA...|
|       ALVAREZ, PAUL|ATOR, GENERAL,ZAR...|
|   AMERICAN SAMURAI/|PAGE, KAREN,DARED...|
|             AMPERE/|QUICKSILVER/PIETR...|
|           ANCESTOR/|RECORDER II,FOUND...|
|ANCIENT ONE/BARON MO|BLOODSTORM | MUTA...|
|    ANDERSSEN, TANYA|KA-ZAR/KEV

In [12]:
df = df.withColumn(
    'connection_size', 
    f.size(
        f.split(f.col('connection'),',')
    )
)

df.orderBy(f.col('connection_size')).desc().show()

AttributeError: 'DataFrame' object has no attribute 'desc'